In [1]:
import warnings
from pathlib import Path

import numpy as np
import prism

from imagematerials.concepts import knowledge_graph
from imagematerials.factory import ModelFactory, Sector
from imagematerials.maintenance import Maintenance
from imagematerials.model import (
    GenericEndOfLife,
    GenericMainModel,
    GenericMaterials,
    GenericStocks,
    MaterialIntensities,
)
from imagematerials.util import (
    export_to_netcdf,
    import_from_netcdf,
    read_circular_economy_config,
    read_climate_policy_config,
    rebroadcast_prep_data,
)

from imagematerials.vehicles import preprocess
from imagematerials.eol import preprocess_eol


In [2]:
prep_eol = preprocess_eol("../../data/raw")

In [3]:
print(prep_eol["collection"].dims)

('Time', 'Region', 'Sector', 'Type', 'material')


In [4]:
list(prep_eol)

['collection', 'reuse', 'recycling']

In [5]:
base_dir = Path("..", "..", "data", "raw")

climate_policy_scenario_dir = base_dir/ 'SSP2'
circular_economy_scenario_dirs = {"slow": base_dir / 'circular_economy_scenarios' / 'slow'}

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    climate_policy_config = read_climate_policy_config(climate_policy_scenario_dir)
    circular_economy_config = read_circular_economy_config(circular_economy_scenario_dirs)

    prep_data = preprocess(base_dir, climate_policy_config, circular_economy_config)

# warnings.simplefilter("ignore")
# prep_data = preprocess()


## run vehicles module

In [6]:
share_coords = set()
for cur_type in prep_data["shares"].Type.values:
    share_coords.add(cur_type.split(" - ")[0])
output_coords_type = [x for x in prep_data["stocks"].Type.values if x not in share_coords] + list(prep_data["shares"].coords["Type"].values)
prep_data.pop("shares")
new_prep_data = rebroadcast_prep_data(prep_data, knowledge_graph, dim="Type", output_coords=output_coords_type)
new_prep_data = rebroadcast_prep_data(new_prep_data, knowledge_graph, dim="Region", output_coords=prep_data["stocks"].coords["Region"].values)
new_prep_data["knowledge_graph"] = knowledge_graph

new_prep_data["weights"] = new_prep_data.pop("vehicle_weights")
vhc_prep_data = new_prep_data.copy()

In [7]:
list(vhc_prep_data)

['lifetimes',
 'battery_materials',
 'battery_shares',
 'material_fractions',
 'battery_weights',
 'stocks',
 'maintenance_material_fractions',
 'knowledge_graph',
 'weights']

In [8]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [9]:
# Define the coordinates of all dimensions.
Region = list(vhc_prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(vhc_prep_data["stocks"].coords["Type"].values)
material = list(vhc_prep_data["material_fractions"].coords["material"].values)

In [11]:
vhc_prep_data['shares'] = None

vhc = Sector(name="vhc", data = vhc_prep_data)
eol = Sector(name="eol", data = prep_eol, 
             coordinates = {"eoltype": ["collected", "reusable", "recyclable", "losses", "surpluslosses"],
                            "Region": Region}
             )

factory = ModelFactory([vhc, eol], complete_timeline)
factory.add(GenericStocks, "vhc")
factory.add(GenericMaterials, "vhc")
factory.add(GenericEndOfLife, "eol", input_sources={
    "outflow_by_cohort_materials": "vhc",
    "collection": "eol",
    "reuse": "eol",
    "recycling": "eol"}
)


model = factory.finish()

AttributeError: 'Sector' object has no attribute 'coordinate_sources'

In [ ]:
model.simulate(simulation_timeline)

/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value
/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value


/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value
/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value


/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value
/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value


/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value
/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value


/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value
/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value


...

/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value
/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value


/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value
/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value


/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value
/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1560: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value


TimeVariable(timeline=Timeline(start=1960, end=2060, stepsize=1), dims=(Dimension(label="Region", coords=[np.str_('1'), np.str_('10'), np.str_('11'), np.str_('12'), np.str_('13'), np.str_('14'), np.str_('15'), np.str_('16'), np.str_('17'), np.str_('18'), np.str_('19'), np.str_('2'), np.str_('20'), np.str_('21'), np.str_('22'), np.str_('23'), np.str_('24'), np.str_('25'), np.str_('26'), np.str_('27'), np.str_('28'), np.str_('3'), np.str_('4'), np.str_('5'), np.str_('6'), np.str_('7'), np.str_('8'), np.str_('9')]), Dimension(label="Type", coords=[np.str_('Bikes'), np.str_('Freight Planes'), np.str_('Freight Trains'), np.str_('High Speed Trains'), np.str_('Inland Ships'), np.str_('Large Ships'), np.str_('Medium Ships'), np.str_('Passenger Planes'), np.str_('Small Ships'), np.str_('Trains'), np.str_('Very Large Ships'), np.str_('Cars - BEV'), np.str_('Cars - FCV'), np.str_('Cars - HEV'), np.str_('Cars - ICE'), np.str_('Cars - PHEV'), np.str_('Cars - Trolley'), np.str_('Heavy Freight Trucks

IndexError: dimension coordinate 'Region' conflicts between indexed and indexing objects:
<xarray.DataArray 'Region' (Region: 0)> Size: 0B
array([], dtype='<U21')
Coordinates:
  * Region   (Region) <U21 0B 
    Time     int64 8B 2020
vs.
<xarray.IndexVariable 'Region' (Region: 26)> Size: 208B
array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
       19, 20, 21, 22, 23, 24, 25, 26])

In [ ]:
Region

[np.str_('1'),
 np.str_('10'),
 np.str_('11'),
 np.str_('12'),
 np.str_('13'),
 np.str_('14'),
 np.str_('15'),
 np.str_('16'),
 np.str_('17'),
 np.str_('18'),
 np.str_('19'),
 np.str_('2'),
 np.str_('20'),
 np.str_('21'),
 np.str_('22'),
 np.str_('23'),
 np.str_('24'),
 np.str_('25'),
 np.str_('26'),
 np.str_('27'),
 np.str_('28'),
 np.str_('3'),
 np.str_('4'),
 np.str_('5'),
 np.str_('6'),
 np.str_('7'),
 np.str_('8'),
 np.str_('9')]

## checking resuls

In [ ]:
outflow_by_cohort_materials = vhc.all_data["outflow_by_cohort_materials"]
print(outflow_by_cohort_materials.to_array().dims)

collection = eol.all_data["collection"]
print(collection.dims)


('time', 'Region', 'Type', 'material')
('Time', 'Region', 'Sector', 'Type', 'material')


In [ ]:
outflow_by_cohort_materials

TimeVariable(
  timeline=Timeline(start=1960, end=2060, stepsize=1),
  unit=count,
  dims=
	Dimension(label="Region", coords=[np.str_('1'), np.str_('10'),
	  np.str_('11'), np.str_('12'), np.str_('13'), np.str_('14'),
	  np.str_('15'), np.str_('16'), np.str_('17'), np.str_('18'),
	  np.str_('19'), np.str_('2'), np.str_('20'), np.str_('21'),
	  np.str_('22'), np.str_('23'), np.str_('24'), np.str_('25'),
	  np.str_('26'), np.str_('27'), np.str_('28'), np.str_('3'), np.str_('4'),
	  np.str_('5'), np.str_('6'), np.str_('7'), np.str_('8'), np.str_('9')])
	  Dimension(label="Type", coords=[np.str_('Bikes'), np.str_('Freight
	  Planes'), np.str_('Freight Trains'), np.str_('High Speed Trains'),
	  np.str_('Inland Ships'), np.str_('Large Ships'), np.str_('Medium
	  Ships'), np.str_('Passenger Planes'), np.str_('Small Ships'),
	  np.str_('Trains'), np.str_('Very Large Ships'), np.str_('Cars - BEV'),
	  np.str_('Cars - FCV'), np.str_('Cars - HEV'), np.str_('Cars - ICE'),
	  np.str_('Cars - PHEV')

In [ ]:
print(outflows_by_cohort_materials.dims)

print(eol.all_data.keys())

print(prep_eol.keys())



NameError: name 'outflows_by_cohort_materials' is not defined

In [ ]:
print(eol.all_data.keys())


dict_keys(['collection', 'reuse', 'recycling', 'eol_materials'])


In [ ]:
list(model.default)

['stocks',
 'lifetimes',
 'shares',
 'knowledge_graph',
 'outflow_by_cohort',
 'inflow',
 'stock_by_cohort',
 'weights',
 'material_fractions',
 'stock_by_cohort_materials',
 'inflow_materials',
 'outflow_by_cohort_materials']

In [ ]:
model.outflow_by_cohort_materials

TimeVariable(
  timeline=Timeline(start=1960, end=2060, stepsize=1),
  unit=count,
  dims=
	Dimension(label="Region", coords=[np.str_('1'), np.str_('10'),
	  np.str_('11'), np.str_('12'), np.str_('13'), np.str_('14'),
	  np.str_('15'), np.str_('16'), np.str_('17'), np.str_('18'),
	  np.str_('19'), np.str_('2'), np.str_('20'), np.str_('21'),
	  np.str_('22'), np.str_('23'), np.str_('24'), np.str_('25'),
	  np.str_('26'), np.str_('27'), np.str_('28'), np.str_('3'), np.str_('4'),
	  np.str_('5'), np.str_('6'), np.str_('7'), np.str_('8'), np.str_('9')])
	  Dimension(label="Type", coords=[np.str_('Bikes'), np.str_('Freight
	  Planes'), np.str_('Freight Trains'), np.str_('High Speed Trains'),
	  np.str_('Inland Ships'), np.str_('Large Ships'), np.str_('Medium
	  Ships'), np.str_('Passenger Planes'), np.str_('Small Ships'),
	  np.str_('Trains'), np.str_('Very Large Ships'), np.str_('Cars - BEV'),
	  np.str_('Cars - FCV'), np.str_('Cars - HEV'), np.str_('Cars - ICE'),
	  np.str_('Cars - PHEV')

## running with EoL

In [ ]:
factory = ModelFactory(vhc_prep_data, complete_timeline)
factory.add(GenericStocks)
factory.add(GenericMaterials)
factory.add(GenericEndOfLife)

model = factory.finish()

ValueError: Cannot find input data 'collection' in namespace 'default' for model class '<class 'imagematerials.model.GenericEndOfLife'>'.

In [ ]:
model.stocks.sel(
    Region=1,
    Type='bikes',
#    material = 'Steel',
).plot(marker='o')


plt.show()

KeyError: 'outflows_by_cohort'